# 10 — Route watch: forecast today, grade yesterday

<!-- contract -->

| | |
|---|---|
| **Reads** | AeroDataBox (airport departures, flight status), FAA NAS, `api_silver_flights` |
| **Writes** | `api_silver_flights` (rows with `row_kind = 'watch'`) |
| **Runtime** | ~2 min |
| **Requires** | Secret `aerodatabox_key`; `07_score` and `08_monitor` run after it |

The scheduled half of the monitoring loop. `06` answers one traveller's question about one
flight; this notebook runs every morning over a few busy routes and does two things:

1. **Collects outcomes** for flights it forecast on earlier days, once the provider reports
   them `Arrived`.
2. **Forecasts today** — a handful of departures per route, spread across the day, each
   scored before it leaves.

`07_score` then scores the run and attaches outcomes to the forecasts they belong to, and
`08_monitor` grades them. This is the only evaluation in the project on flights from after the
training period, and it runs without anyone typing a flight number.


In [0]:
%pip install holidays -q
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import sys
sys.path.append("..")

from datetime import datetime, timezone

import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql import functions as F

from src import config
from src.aerodatabox import AeroDataBoxClient, departure_to_row, flight_to_row
from src.faa_nas import fetch_status, summarise
from src.route_watch import (
    dedupe_same_minute, departure_window, parse_routes, quota_exhausted, spread_pick,
)

## Inputs

Routes default to `WATCH_ROUTES` in `src/config.py` and can be overridden per run. Each call
checks the remaining AeroDataBox units first and stops at `AERODATABOX_QUOTA_RESERVE`, so a
schedule left running can never spend the budget that manual runs of `06` depend on.

In [0]:
dbutils.widgets.text("ROUTES", config.WATCH_ROUTES)
dbutils.widgets.text("MAX_PER_ROUTE", str(config.WATCH_MAX_PER_ROUTE))

ROUTES = parse_routes(dbutils.widgets.get("ROUTES"))
MAX_PER_ROUTE = int(dbutils.widgets.get("MAX_PER_ROUTE"))

no_tz = [origin for origin, _ in ROUTES if origin not in config.WATCH_ORIGIN_TZ]
if no_tz:
    raise ValueError(f"No time zone configured for {no_tz}. Add it to WATCH_ORIGIN_TZ in src/config.py.")

NOW = datetime.now(timezone.utc)
RUN_ID = NOW.strftime("%Y%m%dT%H%M%SZ")
adb = AeroDataBoxClient(
    dbutils.secrets.get(config.AERODATABOX_SECRET_SCOPE, config.AERODATABOX_SECRET_KEY)
)


def budget_left():
    if quota_exhausted(adb.last_quota, config.AERODATABOX_QUOTA_RESERVE):
        print(f"  Stopping: {adb.last_quota.get('units_remaining')} units left, "
              f"reserve is {config.AERODATABOX_QUOTA_RESERVE}.")
        return False
    return True


print(f"Run {RUN_ID}: {', '.join(f'{o}->{d}' for o, d in ROUTES)}, "
      f"up to {MAX_PER_ROUTE} forecasts per route")

Run 20260928T192428Z: ATL->JFK, ORD->LGA, LAX->SFO, up to 3 forecasts per route


## Outcomes for earlier forecasts

The airport feed a forecast is built from carries no arrival time, so the outcome has to come
from the flight itself: one status call per flight, up to `WATCH_LOOKBACK_DAYS` after its date.
A flight is graded only once the provider reports it `Arrived`; until then its row keeps
waiting, and a cancelled flight is dropped from the queue. A flight the provider never marks
`Arrived` within the lookback stays ungraded, and `08` counts it separately.

In [0]:
due = []
if spark.catalog.tableExists(config.API_SILVER):
    silver = spark.table(config.API_SILVER)
    if "row_kind" in silver.columns:
        phase = (F.coalesce(F.col("provider_phase"), F.lit("unknown"))
                 if "provider_phase" in silver.columns else F.lit("unknown"))
        due = (
            silver
            .filter((F.col("row_kind") == "watch") & F.col("arrival_delay").isNull())
            .filter(phase != "cancelled")
            .filter(F.col("flight_date") < F.current_date())
            .filter(F.col("flight_date") >= F.date_sub(F.current_date(), config.WATCH_LOOKBACK_DAYS))
            .select("flight_iata", "flight_date", "origin_airport_code", "destination_airport_code")
            .distinct()
            .orderBy("flight_date", "flight_iata")
            .collect()
        )
print(f"{len(due)} earlier forecast(s) waiting on an outcome")

outcome_rows = []
for d in due:
    if not budget_left():
        break
    try:
        legs = adb.flight_by_number(d["flight_iata"], d["flight_date"])
    except Exception as e:
        print(f"  {d['flight_iata']} {d['flight_date']}: unavailable ({type(e).__name__})")
        continue
    match = [r for r in (flight_to_row(leg) for leg in legs)
             if r and r["origin_airport_code"] == d["origin_airport_code"]
             and r["destination_airport_code"] == d["destination_airport_code"]]
    if not match:
        print(f"  {d['flight_iata']} {d['flight_date']}: no longer served by the provider")
        continue
    row = match[0]
    outcome_rows.append(row)
    arrival = "pending" if row["arrival_delay"] is None else f"{row['arrival_delay']:+.0f} min"
    print(f"  {row['flight_iata']:<8} {row['origin_airport_code']}->{row['destination_airport_code']}  "
          f"{row['flight_date']}  {str(row['flight_status']):<10} arrival {arrival}")

## Today's forecasts

One airport query per route covers the next 12 hours from an hour after now, so every flight
it returns can still be forecast before it leaves. The window is cut into `MAX_PER_ROUTE` equal
bands and one departure is kept from each, to keep the hour-of-day mix. Which flight in a band
rotates with the date, so the watch grades a different set of flights each day instead of the
same few repeatedly.

In [0]:
forecast_rows = []
for origin, dest in ROUTES:
    if not budget_left():
        break
    start, end = departure_window(NOW, config.WATCH_ORIGIN_TZ[origin],
                                  config.ALTERNATIVE_MIN_LEAD_MINUTES, config.WATCH_WINDOW_HOURS)
    try:
        departures = adb.airport_departures(origin, start, end)
    except Exception as e:
        print(f"{origin}->{dest}: departures unavailable ({type(e).__name__}: {e})")
        continue
    on_route = dedupe_same_minute([
        r for r in (departure_to_row(dep, origin) for dep in departures)
        if r and r["destination_airport_code"] == dest
    ])
    picked = spread_pick(on_route, MAX_PER_ROUTE, rotation=start.toordinal())
    carriers = sorted({r.get("airline_name") or r.get("airline_code") or "?" for r in on_route})
    names = ", ".join(f"{r['flight_iata']} {r['crs_dep_time']:04d}" for r in picked) or "nothing"
    print(f"{origin}->{dest}  {start:%H:%M}-{end:%H:%M} local: {len(on_route)} on the route "
          f"({', '.join(carriers) or 'none'}); forecasting {names}")
    forecast_rows += picked

print(f"\nQuota: {adb.last_quota.get('units_remaining')} AeroDataBox units remaining")

## Airspace context, then the write

FAA airspace conditions are recorded beside each forecast as context the model never sees,
so `08_monitor` can later ask whether misses cluster under degraded conditions. Rows are
merged on the flight's identity, the same key `06` uses, so an outcome row updates the
morning's forecast row rather than adding a second one.

In [0]:
nas = None
try:
    nas = fetch_status()
except Exception as e:
    print(f"FAA NAS status unavailable ({type(e).__name__})")

rows = []
for r in outcome_rows + forecast_rows:
    lines = summarise(nas, [r["origin_airport_code"], r["destination_airport_code"]]) if nas else []
    rows.append(dict(
        r,
        row_kind="watch",
        is_flight_of_interest=False,
        flight_phase="unknown",
        observed_departure_hhmm=None,
        observed_spread_minutes=None,
        nas_conditions="; ".join(lines) if lines else None,
        nas_checked_at=nas.updated if nas else None,
        ingest_run_id=RUN_ID,
    ))

MERGE_KEY = ["flight_iata", "flight_date", "origin_airport_code", "destination_airport_code"]
written = 0
if rows:
    sdf = (
        spark.createDataFrame(pd.DataFrame(rows))
        .withColumn("ingested_at", F.current_timestamp())
        .withColumn("fl_number", F.col("fl_number").cast("int"))
        .dropDuplicates(MERGE_KEY)
    )
    if not spark.catalog.tableExists(config.API_SILVER):
        (sdf.limit(0).write.format("delta").mode("overwrite")
         .option("overwriteSchema", "true").saveAsTable(config.API_SILVER))
    else:
        target = {f.name: f.dataType.simpleString() for f in spark.table(config.API_SILVER).schema.fields}
        missing = [f for f in sdf.schema.fields if f.name not in target]
        if missing:
            ddl = ", ".join(f"{f.name} {f.dataType.simpleString().upper()}" for f in missing)
            spark.sql(f"ALTER TABLE {config.API_SILVER} ADD COLUMNS ({ddl})")
        for f in sdf.schema.fields:
            if target.get(f.name) == "void" and f.dataType.simpleString() != "void":
                spark.sql(f"ALTER TABLE {config.API_SILVER} ALTER COLUMN {f.name} "
                          f"TYPE {f.dataType.simpleString().upper()}")

    cols = [f.name for f in sdf.schema.fields]
    (
        DeltaTable.forName(spark, config.API_SILVER).alias("t")
        .merge(sdf.alias("s"), " AND ".join(f"t.{k} = s.{k}" for k in MERGE_KEY))
        .whenMatchedUpdate(set={c: F.col(f"s.{c}") for c in cols})
        .whenNotMatchedInsert(values={c: F.col(f"s.{c}") for c in cols})
        .execute()
    )
    written = sdf.count()
    print(f"Wrote {written} row(s) to {config.API_SILVER}: "
          f"{len(outcome_rows)} outcome check(s), {len(forecast_rows)} new forecast(s)")
    display(
        spark.table(config.API_SILVER).filter(F.col("ingest_run_id") == RUN_ID)
        .select("flight_iata", "origin_airport_code", "destination_airport_code", "flight_date",
                "crs_dep_time", "flight_status", "provider_phase", "dep_delay", "arrival_delay")
        .orderBy("flight_date", "origin_airport_code", "crs_dep_time")
    )
else:
    print("Nothing to write.")

# The job's condition task reads this: 07 and 08 run only if this run wrote rows,
# so an empty morning never re-scores yesterday's forecasts.
try:
    dbutils.jobs.taskValues.set(key="rows_written", value=written)
except Exception:
    pass